## 2 Omnilingual Fientuning

This notebook finetunes the model on Meta's Omnilingual.

In [ ]:
"""
Omnilingual Romansh Fine‑Tuning Script
Launches the official Omnilingual ASR training recipe using your custom dataset.
"""

import os

# Thread limiting
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

import sys
import subprocess
import torch
from pathlib import Path
import yaml

SCRIPT_DIR = Path.cwd().resolve()          # scripts/
ROOT_DIR = SCRIPT_DIR.parent                      # omnilingual/
SUBMODULE_ROOT = ROOT_DIR / "omnilingual_asr"         # submodule root (contains workflows/)

# We need to add the submodule root to PYTHONPATH so the subprocess can find 'workflows'
env = os.environ.copy()
pythonpath = env.get("PYTHONPATH", "")
env["PYTHONPATH"] = str(SUBMODULE_ROOT) + (f":{pythonpath}" if pythonpath else "")

# Also add to our own sys.path for the helper imports (already installed, but we need it for early imports)
sys.path.insert(0, str(ROOT_DIR))

from omnilingual_asr.utils import get_best_gpu
from omnilingual_asr.constants import LANG_DIST_FILE_ROOT, PARQUET_DATA_ROOT

First we need to set the correct dataset path in the `romansh_dataset.yaml` file and the correct path to the stats file in the `romansh-ctc-finetune.yaml` file. The `set_runtime_paths()` function retireves the paths relative to the current directory and sets them in the config files.

In [ ]:
OUTPUT_DIR = "./models/omnilingual-ctc-rm-1b-2"              # where checkpoints/logs go
CONFIG_FILE = SUBMODULE_ROOT / "workflows/recipes/wav2vec2/asr/configs/romansh-ctc-finetune.yaml"
DATASET_CARD = SUBMODULE_ROOT / "src/omnilingual_asr/cards/datasets/romansh_dataset.yaml"
stats_file = LANG_DIST_FILE_ROOT

def set_runtime_paths():
    """Reads the template yaml, updates paths dynamically, and saves the runtime config."""
    if not CONFIG_FILE.exists():
        print(f"Error: Configuration template not found at {CONFIG_FILE}")
        sys.exit(1)
        
    print("Generating runtime YAML configuration dynamically...")
    with open(CONFIG_FILE, "r") as f:
        config_data = yaml.safe_load(f)
    
    resolved_path = LANG_DIST_FILE_ROOT.resolve()
    if "dataset" in config_data and "mixture_parquet_storage_config" in config_data["dataset"]:
        config_data["dataset"]["mixture_parquet_storage_config"]["dataset_summary_path"] = str(resolved_path)
    else:
        print("Error: The template YAML structure does not match the expected nested keys.")
        sys.exit(1)
        
    with open(CONFIG_FILE, "w") as f:
        yaml.dump(config_data, f, default_flow_style=False)
        
    print(f"Runtime config written to {CONFIG_FILE}")
    print(f"   dataset_summary_path dynamically set to: {resolved_path}")

    if not DATASET_CARD.exists():
        print(f"Error: Dataset card template not found at {DATASET_CARD}")
        sys.exit(1)
        
    with open(DATASET_CARD, "r") as f:
        dataset_card_data = yaml.safe_load(f)
        
    if "dataset_config" in dataset_card_data:
        dataset_card_data["dataset_config"]["data"] = str(PARQUET_DATA_ROOT)
    else:
        print("Error: The dataset card template YAML structure does not match the expected keys.")
        sys.exit(1)
        
    with open(DATASET_CARD, "w") as f:
        yaml.dump(dataset_card_data, f, default_flow_style=False)
    print(f"Dataset card config written to: {DATASET_CARD}")
    print(f"    data path set to: {PARQUET_DATA_ROOT}")

set_runtime_paths()


Then we select the optimal GPU for training.

In [ ]:
best_gpu = get_best_gpu()
if torch.cuda.is_available():
    os.environ["CUDA_VISIBLE_DEVICES"] = str(best_gpu)
    print(f"Using GPU {best_gpu}")
else:
    print("No GPU available – falling back to CPU")

Lastly we check that all of the config files are there.

In [ ]:
missing = []
if not DATASET_CARD.exists():
    missing.append(str(DATASET_CARD))
if not stats_file.exists():
    missing.append(str(stats_file))
if not CONFIG_FILE.exists():
    missing.append(str(CONFIG_FILE))

if missing:
    print("Missing required files:")
    for m in missing:
        print(f"  - {m}")
    sys.exit(1)

And then we can launch training.

In [ ]:
cmd = [
    "python", "-m", "workflows.recipes.wav2vec2.asr",
    OUTPUT_DIR,
    "--config-file", str(CONFIG_FILE),
]

print("=" * 60)
print("Starting Omnilingual fine‑tuning")
print(f"   Output dir : {OUTPUT_DIR}")
print(f"   Config     : {CONFIG_FILE}")
print(f"   GPU        : {best_gpu if torch.cuda.is_available() else 'CPU'}")
print("=" * 60)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,                    # PYTHONPATH set here
    cwd=str(ROOT_DIR),
)

for line in process.stdout:
    print(line, end="")

process.wait()

if process.returncode == 0:
    print(f"\nTraining finished. Model saved in: {OUTPUT_DIR}")
else:
    print(f"\nTraining failed with code {process.returncode}")
    sys.exit(process.returncode)